# Neural network training PyTorch

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use project root relative to this notebook
BASE_DIR = Path('/home/amraas/projects/realestatecons')
DATA_DIR = BASE_DIR / 'data' / 'raw'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

In [ ]:
test_df_id = test_df['Id'] # saving id for submission

In [2]:
import sys

sys.path.append(str(BASE_DIR))

from src.data.preprocess import clean_test_data, clean_train_data
from src.features.features import (
    add_engineered_features,
    add_log_transformed_features,
    drop_highly_correlated_features,
)

In [3]:
train_df_cleaned = clean_train_data(train_df)
test_df_cleaned = clean_test_data(test_df, train_df)

train_df_features = add_engineered_features(train_df_cleaned)
test_df_features = add_engineered_features(test_df_cleaned)

In [4]:
train_df = drop_highly_correlated_features(train_df_features)
test_df = drop_highly_correlated_features(test_df_features)

In [5]:
train_df.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath', 'BsmtHalfBath',
       'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond',
       'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3Ss

## splitting

In [21]:
# PyTorch setup
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import itertools
import math


def set_seed(seed: int = 42) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prepare features/targets
TARGET_COL = "SalePrice"

train_df_nn = train_df.copy()
test_df_nn = test_df.copy()

y = train_df_nn[TARGET_COL].to_numpy(dtype=np.float32)
X = train_df_nn.drop(columns=[TARGET_COL])

X = pd.get_dummies(X, drop_first=True)
test_X = pd.get_dummies(test_df_nn, drop_first=True)

X, test_X = X.align(test_X, join="left", axis=1, fill_value=0)

X_np = X.to_numpy(dtype=np.float32)
test_X_np = test_X.to_numpy(dtype=np.float32)

## Grid search with K-folds (Adam + MSE)
We tune batch size and learning rate using 5-fold cross-validation, then retrain on the full training data and generate a submission file.

In [11]:
class MLPRegressor(nn.Module):
    def __init__(self, in_features: int, hidden_sizes=(256, 128), dropout: float = 0.1):
        super().__init__()
        layers = []
        last = in_features
        for size in hidden_sizes:
            layers.append(nn.Linear(last, size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            last = size
        layers.append(nn.Linear(last, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def init_he(m: nn.Module) -> None:
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)


def train_one_fold(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    batch_size: int,
    learning_rate: float,
    epochs: int,
    use_he_init: bool,
) -> float:
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    train_ds = TensorDataset(
        torch.from_numpy(X_train_s),
        torch.from_numpy(y_train).unsqueeze(1),
    )
    val_ds = TensorDataset(
        torch.from_numpy(X_val_s),
        torch.from_numpy(y_val).unsqueeze(1),
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = MLPRegressor(in_features=X_train_s.shape[1]).to(device)
    if use_he_init:
        model.apply(init_he)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy().squeeze()
            preds.append(pred)
            targets.append(yb.numpy().squeeze())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    rmse = math.sqrt(mean_squared_error(targets, preds))
    return rmse


def cross_validate(
    X_data: np.ndarray,
    y_data: np.ndarray,
    batch_size: int,
    learning_rate: float,
    epochs: int,
    use_he_init: bool,
    n_splits: int = 5,
) -> tuple[float, float]:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses = []

    for train_idx, val_idx in kf.split(X_data):
        rmse = train_one_fold(
            X_data[train_idx],
            y_data[train_idx],
            X_data[val_idx],
            y_data[val_idx],
            batch_size=batch_size,
            learning_rate=learning_rate,
            epochs=epochs,
            use_he_init=use_he_init,
        )
        rmses.append(rmse)

    return float(np.mean(rmses)), float(np.std(rmses))


# Grid search over main hyperparameters
batch_sizes = [2, 4, 8, 16]
learning_rates = [1e-2, 1e-3, 1e-4]
base_epochs = 20

results = []
for bs, lr in itertools.product(batch_sizes, learning_rates):
    mean_rmse, std_rmse = cross_validate(
        X_np,
        y,
        batch_size=bs,
        learning_rate=lr,
        epochs=base_epochs,
        use_he_init=True,
        n_splits=5,
    )
    results.append(
        {
            "batch_size": bs,
            "learning_rate": lr,
            "mean_rmse": mean_rmse,
            "std_rmse": std_rmse,
        }
    )

results_df = pd.DataFrame(results).sort_values("mean_rmse")
results_df

,batch_size,learning_rate,mean_rmse,std_rmse
9,16,0.0100,43929.440326,6650.180917
3,4,0.0100,45419.129745,6436.164600
6,8,0.0100,46036.299017,9525.129292
0,2,0.0100,46375.087522,8192.032014
7,8,0.0010,48226.653188,3092.824588
4,4,0.0010,49496.920532,3369.787207
1,2,0.0010,50539.158289,3635.074620
10,16,0.0010,51151.608418,4816.788911
2,2,0.0001,83954.343894,4893.313693
5,4,0.0001,123498.273361,5039.780636


In [ ]:
test_df_nn['Id'] = test_df_id

In [23]:
best_params = results_df.iloc[0].to_dict()

print("Best params (He init):", best_params)

he_mean_rmse, he_std_rmse = cross_validate(
    X_np,
    y,
    batch_size=int(best_params["batch_size"]),
    learning_rate=float(best_params["learning_rate"]),
    epochs=base_epochs,
    use_he_init=True,
    n_splits=5,
)

base_mean_rmse, base_std_rmse = cross_validate(
    X_np,
    y,
    batch_size=int(best_params["batch_size"]),
    learning_rate=float(best_params["learning_rate"]),
    epochs=base_epochs,
    use_he_init=False,
    n_splits=5,
)

print(
    f"He init CV RMSE: {he_mean_rmse:.4f} ± {he_std_rmse:.4f} | "
    f"Default init CV RMSE: {base_mean_rmse:.4f} ± {base_std_rmse:.4f}"
)

# Train final model on full data with best hyperparameters (He init)
final_batch_size = int(best_params["batch_size"])
final_lr = float(best_params["learning_rate"])

scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(X_np)
X_test_full = scaler_full.transform(test_X_np)

train_ds = TensorDataset(
    torch.from_numpy(X_full),
    torch.from_numpy(y).unsqueeze(1),
)
train_loader = DataLoader(train_ds, batch_size=final_batch_size, shuffle=True)

final_model = MLPRegressor(in_features=X_full.shape[1]).to(device)
final_model.apply(init_he)

optimizer = torch.optim.Adam(final_model.parameters(), lr=final_lr)
criterion = nn.MSELoss()

for _ in range(base_epochs):
    final_model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = final_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

final_model.eval()
with torch.no_grad():
    test_preds = final_model(torch.from_numpy(X_test_full).to(device)).cpu().numpy().squeeze()



Best params (He init): {'batch_size': 16.0, 'learning_rate': 0.01, 'mean_rmse': 43929.44032612752, 'std_rmse': 6650.180917328585}
He init CV RMSE: 43519.8547 ± 5561.7808 | Default init CV RMSE: 45726.9100 ± 6972.8581


In [22]:
submission = pd.DataFrame(
    {
        "Id": test_df_nn["Id"].to_numpy(),
        "SalePrice": np.maximum(test_preds, 0),
    }
)

submission_path = BASE_DIR / "reports" / "submissions" / "neural_net_gridsearch.csv"
submission.to_csv(submission_path, index=False)
submission_path

PosixPath('/home/amraas/projects/realestatecons/reports/submissions/neural_net_gridsearch.csv')